In [38]:
import pandas as pd

df_telemetry = pd.read_csv('aviation_data/fact_telemetry_high_freq.csv')
df_summary = pd.read_csv('aviation_data/flight_summary.csv')
df_registry = pd.read_csv('aviation_data/dim_aircraft.csv')

# 1. Merge / "JOIN" 
# 1.1 Telemetry to mission details
master_df = pd.merge(df_telemetry, df_summary, on='flight_id', how='left')

# 1.2 Mission to specifications
master_df = pd.merge(master_df, df_registry, on='tail_number', how='left')

master_df.head()

,flight_id,timestamp_sec,altitude_ft,battery_soc_pct,phase,tail_number,payload_lbs,ambient_temp_c,ticket_revenue,model,battery_capacity_kwh,max_payload_lbs,firmware_version
0,FL-00001,0,0.0,99.97,CLIMB,N103AL,1959.212357,35.86111,4862.392073,Alice_V1,850,2500,v2.1
1,FL-00001,1,33.3,99.94,CLIMB,N103AL,1959.212357,35.86111,4862.392073,Alice_V1,850,2500,v2.1
2,FL-00001,2,66.6,99.91,CLIMB,N103AL,1959.212357,35.86111,4862.392073,Alice_V1,850,2500,v2.1
3,FL-00001,3,99.9,99.88,CLIMB,N103AL,1959.212357,35.86111,4862.392073,Alice_V1,850,2500,v2.1
4,FL-00001,4,133.2,99.85,CLIMB,N103AL,1959.212357,35.86111,4862.392073,Alice_V1,850,2500,v2.1


In [39]:
# 2. Missingness & Outliers
# 2.1 Maintain time-series continuity : if a sensor drops out for a second, assume it remains the same as  previous second
master_df['battery_soc_pct'] = master_df['battery_soc_pct'].ffill()

# 2.2 Battery cannot be > 100% or < 0%. Altitude cannot be negative.
master_df['battery_soc_pct'] = master_df['battery_soc_pct'].clip(0, 100)
master_df['altitude_ft'] = master_df['altitude_ft'].clip(lower=0)

In [40]:
{master_df['altitude_ft'].min()}

{np.float64(0.0)}

In [41]:
{master_df['battery_soc_pct'].isnull().sum()}

{np.int64(0)}

In [42]:
master_df[['flight_id', 'timestamp_sec', 'battery_soc_pct', 'altitude_ft']].head()

,flight_id,timestamp_sec,battery_soc_pct,altitude_ft
0,FL-00001,0,99.97,0.0
1,FL-00001,1,99.94,33.3
2,FL-00001,2,99.91,66.6
3,FL-00001,3,99.88,99.9
4,FL-00001,4,99.85,133.2


In [43]:
# 3 Aggregate to 1-minute intervals for business decision
master_df['timestamp_min'] = (master_df['timestamp_sec'] // 60) 

# 3.2 Group by Flight and Minute, taking the Mean for metrics
analytical_view = master_df.groupby(['flight_id', 'tail_number', 'timestamp_min', 
                                   'payload_lbs', 'ambient_temp_c', 'firmware_version']).agg({
    'battery_soc_pct': 'mean',
    'altitude_ft': 'mean',
    'ticket_revenue': 'first' 
}).reset_index()

analytical_view.to_csv('aviation_data/Alice_Master_Analysis_View.csv', index=False)
analytical_view.head(10)

,flight_id,tail_number,timestamp_min,payload_lbs,ambient_temp_c,firmware_version,battery_soc_pct,altitude_ft,ticket_revenue
0,FL-00001,N103AL,0,1959.212357,35.86111,v2.1,99.0850,982.350,4862.392073
1,FL-00001,N103AL,1,1959.212357,35.86111,v2.1,97.2850,2980.350,4862.392073
2,FL-00001,N103AL,2,1959.212357,35.86111,v2.1,95.4850,4978.350,4862.392073
3,FL-00001,N103AL,3,1959.212357,35.86111,v2.1,93.6850,6976.350,4862.392073
4,FL-00001,N103AL,4,1959.212357,35.86111,v2.1,91.8850,8974.350,4862.392073
5,FL-00001,N103AL,5,1959.212357,35.86111,v2.1,90.0850,10972.350,4862.392073
6,FL-00001,N103AL,6,1959.212357,35.86111,v2.1,88.2850,12970.350,4862.392073
7,FL-00001,N103AL,7,1959.212357,35.86111,v2.1,86.5845,14734.425,4862.392073
8,FL-00001,N103AL,8,1959.212357,35.86111,v2.1,85.4690,15000.000,4862.392073
9,FL-00001,N103AL,9,1959.212357,35.86111,v2.1,84.4490,15000.000,4862.392073


In [44]:
analytical_view.columns.tolist()

['flight_id',
 'tail_number',
 'timestamp_min',
 'payload_lbs',
 'ambient_temp_c',
 'firmware_version',
 'battery_soc_pct',
 'altitude_ft',
 'ticket_revenue']